# EDA and join new dataset
- fetched data from kaggle with 600k tracks (spotify)
- join two different files

In [7]:
import os
import pandas as pd
print(os.getcwd())
print(os.listdir('../data/raw'))

df = pd.read_csv('../data/raw/tracks.csv')


c:\Users\ricka\Documents\Course\examensarbete\eda
['artists.csv', 'dict_artists.json', 'tracks.csv']


In [8]:
df.columns.tolist()

['id',
 'name',
 'popularity',
 'duration_ms',
 'explicit',
 'artists',
 'id_artists',
 'release_date',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature']

In [9]:
df["spotify_url"] = (
    "https://open.spotify.com/track/" + df["id"].astype(str)
)

display(
    df[["name", "artists", "id", "spotify_url"]].head(10)
)

,name,artists,id,spotify_url
0,Carve,['Uli'],35iwgR4jXetI318WEWsa1Q,https://open.spotify.com/track/35iwgR4jXetI318...
1,Capítulo 2.16 - Banquero Anarquista,['Fernando Pessoa'],021ht4sdgPcrDgSk7JTbKY,https://open.spotify.com/track/021ht4sdgPcrDgS...
2,Vivo para Quererte - Remasterizado,['Ignacio Corsini'],07A5yehtSnoedViJAZkNnc,https://open.spotify.com/track/07A5yehtSnoedVi...
3,El Prisionero - Remasterizado,['Ignacio Corsini'],08FmqUhxtyLTn6pAh6bk45,https://open.spotify.com/track/08FmqUhxtyLTn6p...
4,Lady of the Evening,['Dick Haymes'],08y9GfoqCWfOGsKdwojr5e,https://open.spotify.com/track/08y9GfoqCWfOGsK...
5,Ave Maria,['Dick Haymes'],0BRXJHRNGQ3W4v9frnSfhu,https://open.spotify.com/track/0BRXJHRNGQ3W4v9...
6,La Butte Rouge,['Francis Marty'],0Dd9ImXtAtGwsmsAD69KZT,https://open.spotify.com/track/0Dd9ImXtAtGwsms...
7,La Java,['Mistinguett'],0IA0Hju8CAgYfV1hwhidBH,https://open.spotify.com/track/0IA0Hju8CAgYfV1...
8,Old Fashioned Girl,['Greg Fieler'],0IgI1UCz84pYeVetnl1lGP,https://open.spotify.com/track/0IgI1UCz84pYeVe...
9,Martín Fierro - Remasterizado,['Ignacio Corsini'],0JV4iqw2lSKJaHBQZ0e5zK,https://open.spotify.com/track/0JV4iqw2lSKJaHB...


In [10]:
from IPython.display import IFrame

track_id = df.iloc[20]["id"]
embed_url = f"https://open.spotify.com/embed/track/{track_id}"

IFrame(embed_url, width=300, height=80)

In [11]:
df_artists = pd.read_csv('../data/raw/artists.csv')

In [12]:
# Artists: share with empty genre at different popularity thresholds
for t in (30, 50, 70):
    sub = df_artists[df_artists["popularity"] >= t]
    print(t, len(sub), (sub["genres"] == "[]").mean())

30 116406 0.2855522911190145
50 22897 0.11743896580338035
70 1972 0.028904665314401622


In [13]:
import ast

# Parse genres (text) into a list; a malformed value returns an empty list
def parse_genres(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

# Only artists with popularity >= 50
popular = df_artists[df_artists["popularity"] >= 50].copy()
popular["genres_list"] = popular["genres"].apply(parse_genres)

In [14]:
# Genre distribution: explode so each genre gets its own row
genre_counts = popular["genres_list"].explode().value_counts()
print(len(genre_counts))
print(genre_counts.head(20))
print((genre_counts < 5).sum())  # genres with fewer than 5 artists

3265
genres_list
pop               564
rock              546
dance pop         541
rap               422
edm               414
pop rap           405
hip hop           398
modern rock       392
latin             381
electro house     369
pop rock          358
pop dance         355
electropop        352
post-teen pop     333
tropical house    319
indie pop         312
soft rock         296
trap              293
mellow gold       289
indie rock        286
Name: count, dtype: int64
1661


In [15]:
# broad_map: broad genre -> words to look for
broad_map = {
    "pop": ["pop"],
    "rock": ["rock"],
    "hip hop": ["hip hop", "rap", "drill"],
    "electronic": ["house", "edm", "electro", "rawstyle"],
    "latin": ["latin", "mexican", "corrido", "ranchera", "norteno",
              "banda", "sierreno", "cumbia"],
    "brazilian": ["mpb", "pagode", "bossa nova", "forro", "samba", "axe"],
    "soundtrack": ["soundtrack", "show tunes", "movie tunes",
                   "hollywood", "broadway", "scorecore"],
    "jazz": ["jazz"], "metal": ["metal"], "country": ["country"],
    "r&b": ["r&b"], "soul": ["soul"], "folk": ["folk"], "indie": ["indie"],
    "punk": ["punk"], "reggae": ["reggae"], "blues": ["blues"],
    "classical": ["classical"], "funk": ["funk"],
}

# exclude: genres that do not fit a quiz
exclude = ["comedy", "hoerspiel", "lo-fi", "chillhop", "sleep",
           "background", "lullaby", "calming", "focus", "reading",
           "environmental", "meditation", "children's"]

# All broad genres for one artist
def broad_genres(genres):
    return sorted({name for name, words in broad_map.items()
                   for g in genres for w in words if w in g})

# True if ALL of the artist's genres are on the exclude list
def is_excluded(genres):
    return len(genres) > 0 and all(any(e in g for e in exclude) for g in genres)

In [16]:
popular["broad"] = popular["genres_list"].apply(broad_genres)
popular["excluded"] = popular["genres_list"].apply(is_excluded)

has_genre = popular["genres_list"].str.len() > 0
keep = popular[has_genre & ~popular["excluded"]].copy()

# Unmapped artists get "other"
keep["broad"] = keep["broad"].apply(lambda b: b if b else ["other"])

print(len(keep))                                          # 19430
print((keep["broad"].apply(lambda b: b != ["other"])).mean())  # ~0.895

19430
0.8947503860010293
